# PySpark Data Processing Pipeline: TPC-H Sales & Customer Analysis

**Big picture:** this notebook builds a distributed pipeline on a multi-GB sales dataset and shows *why* each design choice makes Spark faster.

| Section | What it shows |
|---|---|
| 1. Setup & load | Reading Delta/Parquet tables, dataset size |
| 2. Transformations | Early filters, column pruning, `withColumn`, 5 joins (incl. broadcast), `groupBy` |
| 3. SQL queries | Three Spark SQL queries on a temp view |
| 4. Performance analysis | `.explain()`, naive vs optimized pipeline, timings |
| 5. Caching | `.cache()` for repeated actions |
| 6. Transformations vs actions | Lazy vs eager evaluation |
| 7. Write results | Partitioned Parquet output + partition pruning on read-back |
| 8. MLlib (optional) | KMeans customer segmentation |

**Environment:** Databricks Free Edition (serverless compute, environment version 4+).
**Data:** `samples.tpch`, built into every Databricks workspace, no download needed.

## 1. Setup & load

**Why:** keep all paths/constants in one place, and use a small `timed()` helper so every performance claim in the README is backed by a measured number.

In [ ]:
import time
from pyspark.sql import functions as F

CATALOG = "workspace"          # default catalog in Databricks Free Edition
SCHEMA = "default"
VOLUME = "pyspark_pipeline_output"
OUTPUT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# A Unity Catalog volume is where serverless notebooks can write files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

def timed(label, fn):
    # Run fn(), print and return (result, seconds)
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.2f} s")
    return result, elapsed

print("Output path:", OUTPUT_PATH)

### Load the tables

TPC-H is the industry-standard decision-support benchmark: a wholesale supplier's sales database.
`spark.read.table()` is a **transformation**: it only records *where* the data is. Nothing is read yet.

In [ ]:
TABLES = ["lineitem", "orders", "customer", "nation", "region", "part", "partsupp"]
raw = {t: spark.read.table(f"samples.tpch.{t}") for t in TABLES}

raw["lineitem"].printSchema()

### How big is the data?

**Why:** the assignment requires 1GB+. `DESCRIBE DETAIL` reads the Delta metadata (file count and bytes on disk) without scanning the data.

In [ ]:
size_rows = []
for t in TABLES:
    try:
        d = spark.sql(f"DESCRIBE DETAIL samples.tpch.{t}").select("numFiles", "sizeInBytes").first()
        num_files, size_gb = d["numFiles"], round(d["sizeInBytes"] / 1024**3, 3)
    except Exception as e:
        num_files, size_gb = None, None
        print(f"DESCRIBE DETAIL not available for {t}: {type(e).__name__}")
    size_rows.append((t, raw[t].count(), num_files, size_gb))

size_df = spark.createDataFrame(size_rows, "table string, row_count long, num_files long, size_gb_on_disk double")
display(size_df)

total_gb = sum(r[3] for r in size_rows if r[3] is not None)
total_rows = sum(r[1] for r in size_rows)
print(f"Total: {total_rows:,} rows, {total_gb:.2f} GB on disk (compressed)")

## 2. Transformations: the optimized pipeline

**Goal:** an "enriched sales" table with one row per shipped line item, plus customer, nation, region, part and supplier cost, then revenue, **profit** and delivery metrics.

Optimization choices, and why:

| Choice | Why it helps |
|---|---|
| `select()` only needed columns first | **Column pruning**: Parquet is columnar, so unused columns are never read from disk |
| Filter on **raw** columns (`l_shipdate BETWEEN ...`) | Simple column-vs-literal predicates can be **pushed down** into the scan to skip files/row groups. A filter on `year(l_shipdate)` could not |
| Filter **both** sides of a join (orders by date too) | Fewer rows go into the shuffle |
| `broadcast()` the tiny nation/region table | Sends a copy to every executor, so the big table is **not shuffled** for that join |

In [ ]:
START_DATE, END_DATE = "1995-01-01", "1997-12-31"

# Line items: the large fact table
lineitem = (
    raw["lineitem"]
    .select("l_orderkey", "l_partkey", "l_suppkey", "l_quantity", "l_extendedprice", "l_discount", "l_tax",
            "l_returnflag", "l_shipdate", "l_commitdate", "l_receiptdate", "l_shipmode")
    .filter(F.col("l_shipdate").between(START_DATE, END_DATE))      # FILTER 1: date range on raw column
    .filter(F.col("l_returnflag") != "R")                             # FILTER 2: drop returned items
    .withColumn("net_revenue", F.col("l_extendedprice") * (1 - F.col("l_discount")))
    .withColumn("ship_year", F.year("l_shipdate"))
    .withColumn("delivery_days", F.datediff("l_receiptdate", "l_shipdate"))
    .withColumn("is_late", (F.col("l_receiptdate") > F.col("l_commitdate")).cast("int"))
)

# Orders: in TPC-H an item ships 1-121 days after the order date,
# so this date filter is safe and shrinks the orders side of the join
orders = (
    raw["orders"]
    .select("o_orderkey", "o_custkey", "o_orderdate", "o_orderpriority")
    .filter(F.col("o_orderdate").between("1994-09-01", END_DATE))   # FILTER 3
)

# Customers in good standing only (non-negative account balance)
customer = (
    raw["customer"]
    .select("c_custkey", "c_nationkey", "c_mktsegment", "c_acctbal")
    .filter(F.col("c_acctbal") >= 0)                                 # FILTER 4
)

# Supplier cost per (part, supplier): needed to compute profit
partsupp = raw["partsupp"].select("ps_partkey", "ps_suppkey", "ps_supplycost")

# Part catalogue: p_type looks like "PROMO BURNISHED COPPER" -> last word is the material
part = (
    raw["part"]
    .select("p_partkey", "p_type")
    .withColumn("part_material", F.element_at(F.split("p_type", " "), -1))
    .drop("p_type")
)

# Tiny lookup: 25 nations joined to 5 regions
nation_region = (
    raw["nation"].select("n_nationkey", "n_name", "n_regionkey")
    .join(raw["region"].select("r_regionkey", "r_name"),
          F.col("n_regionkey") == F.col("r_regionkey"))
    .select("n_nationkey", "n_name", "r_name")
)

### Joins

- `lineitem ⋈ orders`: both are large, so Spark uses a **shuffle (sort-merge) join**. This is the main shuffle in the pipeline.
- `⋈ customer`: medium-sized. Adaptive Query Execution (AQE) may switch it to a broadcast join at runtime.
- `⋈ partsupp`: large, joined on a **composite key** (part + supplier), so it's another shuffle join.
- `⋈ part`: medium-sized. AQE may broadcast it.
- `⋈ nation_region`: 25 rows, so we **force a broadcast** with `F.broadcast()`.

In [ ]:
enriched = (
    lineitem
    .join(orders, F.col("l_orderkey") == F.col("o_orderkey"), "inner")
    .join(customer, F.col("o_custkey") == F.col("c_custkey"), "inner")
    .join(partsupp, (F.col("l_partkey") == F.col("ps_partkey")) &
                    (F.col("l_suppkey") == F.col("ps_suppkey")), "inner")
    .join(part, F.col("l_partkey") == F.col("p_partkey"), "inner")
    .join(F.broadcast(nation_region), F.col("c_nationkey") == F.col("n_nationkey"), "inner")
    .withColumn("profit", F.col("net_revenue") - F.col("ps_supplycost") * F.col("l_quantity"))
)

enriched.createOrReplaceTempView("enriched_sales")   # used by the SQL queries below

### GroupBy with aggregations

Revenue and delivery performance by **region × year × market segment**.
`groupBy` causes a shuffle (rows with the same key must be on the same machine). Spark first does a **partial aggregation** on each partition, so only small partial sums are shuffled, not raw rows.

In [ ]:
region_year_agg = (
    enriched
    .groupBy("r_name", "ship_year", "c_mktsegment")
    .agg(
        F.round(F.sum("net_revenue") / 1e6, 2).alias("revenue_millions"),
        F.round(F.sum("profit") / 1e6, 2).alias("profit_millions"),
        F.count("*").alias("line_items"),
        F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
        F.round(F.avg("is_late") * 100, 2).alias("late_pct"),
    )
)

# display() is an ACTION: this is the first time the pipeline actually runs
region_year_summary = region_year_agg.orderBy("r_name", "ship_year", "c_mktsegment")
_, t_main = timed("Main aggregation", lambda: region_year_summary.collect())
display(region_year_summary)

## 3. SQL queries

The DataFrame API and Spark SQL compile to the **same** optimizer (Catalyst), so performance is identical. SQL is just often easier to read for window functions.

### SQL query 1: top 2 nations by revenue in each region, per year

Uses a **window function** (`RANK() OVER (PARTITION BY ...)`).

In [ ]:
top_nations = spark.sql('''
    WITH nation_rev AS (
        SELECT r_name AS region,
               n_name AS nation,
               ship_year,
               ROUND(SUM(net_revenue) / 1e6, 2) AS revenue_millions
        FROM enriched_sales
        GROUP BY r_name, n_name, ship_year
    )
    SELECT *
    FROM (
        SELECT *, RANK() OVER (PARTITION BY region, ship_year
                               ORDER BY revenue_millions DESC) AS revenue_rank
        FROM nation_rev
    )
    WHERE revenue_rank <= 2
    ORDER BY ship_year, region, revenue_rank
''')
display(top_nations)

### SQL query 2: delivery performance by shipping mode

In [ ]:
ship_mode_perf = spark.sql('''
    SELECT l_shipmode                       AS ship_mode,
           COUNT(*)                         AS line_items,
           ROUND(AVG(delivery_days), 2)     AS avg_delivery_days,
           ROUND(100 * AVG(is_late), 2)     AS late_pct,
           ROUND(SUM(net_revenue) / 1e6, 2) AS revenue_millions
    FROM enriched_sales
    GROUP BY l_shipmode
    ORDER BY late_pct DESC
''')
display(ship_mode_perf)

### SQL query 3: profit margin by part material

Uses the `partsupp` supply cost joined in Section 2: profit = revenue − (supply cost × quantity).

In [ ]:
material_profit = spark.sql('''
    SELECT part_material,
           ROUND(SUM(net_revenue) / 1e6, 2)               AS revenue_millions,
           ROUND(SUM(profit) / 1e6, 2)                    AS profit_millions,
           ROUND(100 * SUM(profit) / SUM(net_revenue), 2) AS profit_margin_pct
    FROM enriched_sales
    GROUP BY part_material
    ORDER BY profit_margin_pct DESC
''')
display(material_profit)

## 4. Performance analysis

### 4a. Physical plan of the optimized pipeline

Read the plan **bottom-up** (scan first). Things to look for and screenshot:

| Look for | Meaning |
|---|---|
| `Scan` / `PhotonScan ... samples.tpch.lineitem` with `PushedFilters` / `DataFilters` | Filters pushed into the file scan |
| `ReadSchema: struct<...>` | Only the selected columns are read (**column pruning**) |
| `Exchange hashpartitioning(...)` / `ShuffleExchange` | A **shuffle**, the expensive step |
| `BroadcastHashJoin` / `BroadcastExchange` | The nation/region join (and maybe part/customer via AQE), no shuffle of the big side |
| `SortMergeJoin` or `ShuffledHashJoin` | lineitem ⋈ orders and ⋈ partsupp, the big shuffle joins |
| `HashAggregate` appearing twice (partial + final) | Partial aggregation before the shuffle |
| `AdaptiveSparkPlan isFinalPlan=false` | AQE can still change the plan at runtime |

(On Databricks serverless with Photon, node names start with `Photon...`, e.g. `PhotonBroadcastHashJoin`.)

In [ ]:
region_year_agg.explain(mode="formatted")

### 4b. Naive vs optimized pipeline

**Why:** to measure what the optimizations are worth, the same result is built *badly*:
- joins everything **first**, filters **last**
- filters on a **derived** column (`year(l_shipdate)`) instead of the raw date, which blocks scan-level pushdown
- no date filter on `orders`, no explicit `select()`, no broadcast hint

**Expected insight:** Catalyst automatically fixes *some* of this. It pushes `l_returnflag != 'R'` and `c_acctbal >= 0` below the joins and prunes unused columns even without `select()`. What it **cannot** fix is the predicate on `year(...)`, and the missing filter on `orders`. Those show up as more rows scanned and shuffled.

In [ ]:
def build_naive():
    li, o, c, n, r, p, ps = (raw[t] for t in TABLES)
    joined = (
        li.join(o, li.l_orderkey == o.o_orderkey)
          .join(c, o.o_custkey == c.c_custkey)
          .join(ps, (li.l_partkey == ps.ps_partkey) & (li.l_suppkey == ps.ps_suppkey))
          .join(p, li.l_partkey == p.p_partkey)
          .join(n, c.c_nationkey == n.n_nationkey)
          .join(r, n.n_regionkey == r.r_regionkey)
    )
    return (
        joined
        .withColumn("net_revenue", F.col("l_extendedprice") * (1 - F.col("l_discount")))
        .withColumn("ship_year", F.year("l_shipdate"))
        .withColumn("delivery_days", F.datediff("l_receiptdate", "l_shipdate"))
        .withColumn("is_late", (F.col("l_receiptdate") > F.col("l_commitdate")).cast("int"))
        .withColumn("profit", F.col("net_revenue") - F.col("ps_supplycost") * F.col("l_quantity"))
        .filter(F.col("ship_year").between(1995, 1997))   # filter on DERIVED column, applied last
        .filter(F.col("l_returnflag") != "R")
        .filter(F.col("c_acctbal") >= 0)
        .groupBy("r_name", "ship_year", "c_mktsegment")
        .agg(
            F.round(F.sum("net_revenue") / 1e6, 2).alias("revenue_millions"),
            F.round(F.sum("profit") / 1e6, 2).alias("profit_millions"),
            F.count("*").alias("line_items"),
            F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
            F.round(F.avg("is_late") * 100, 2).alias("late_pct"),
        )
    )

naive_agg = build_naive()
naive_agg.explain(mode="formatted")

In [ ]:
# Time both versions. Each runs twice: the first run can be slowed by cold
# storage reads, so the second run is the fairer comparison.
timings = []
for run in (1, 2):
    _, t_naive = timed(f"Naive     (run {run})", lambda: naive_agg.collect())
    _, t_opt   = timed(f"Optimized (run {run})", lambda: region_year_agg.collect())
    timings += [("naive", run, t_naive), ("optimized", run, t_opt)]

timing_df = spark.createDataFrame(timings, "pipeline string, run int, seconds double")
display(timing_df)

# Sanity check: both pipelines must cover the same groups and the same line items
def fingerprint(df):
    return df.agg(F.count("*").alias("groups"), F.sum("line_items").alias("line_items")).first()

print("Naive     :", fingerprint(naive_agg))
print("Optimized :", fingerprint(region_year_agg))

### 4c. Query Details / Query Profile (screenshots)

Serverless compute has no classic Spark UI. Use the **Query Profile** instead:
1. Under the timing cell's output, click **"See performance"** → pick the query → **"See query profile"**.
2. Screenshot the graph for the **naive** query and the **optimized** query.
3. Click the `lineitem` **scan** node and note **rows read**, **files read / pruned**, and **bytes read**.
4. Click the **shuffle/exchange** nodes and note **bytes shuffled**.

These numbers go in the README's performance analysis.

### 4d. Partitioning

**Why it matters:** each partition is one task. Too few means idle cores; too many means scheduling overhead and tiny files.
- **Input:** partitions follow the Delta files (roughly one task per file chunk).
- **Shuffle:** AQE **coalesces** small shuffle partitions automatically after each shuffle.
- **Output:** we `repartition("ship_year")` before writing, so each year folder gets a few large files instead of many tiny ones (Section 7).

In [ ]:
def num_partitions(df):
    # Count partitions without the RDD API (RDDs are not available on serverless)
    return df.select(F.spark_partition_id().alias("pid")).distinct().count()

print("lineitem input partitions        :", num_partitions(raw["lineitem"]))
print("filtered lineitem partitions     :", num_partitions(lineitem))
try:
    print("AQE enabled                      :", spark.conf.get("spark.sql.adaptive.enabled"))
except Exception:
    print("AQE setting not readable on this compute (enabled by default on Databricks)")

## 5. Caching (bonus)

**Why:** every action re-runs the whole lineage (scan → filter → join → aggregate). If one DataFrame feeds **several** actions (here: three summaries plus the ML model), `.cache()` computes it **once** and keeps it in memory.

`customer_features` is a good candidate: it's the output of the expensive joins, it's small (one row per customer), and it's reused.

In [ ]:
customer_features = (
    enriched
    .groupBy("c_custkey", "c_mktsegment", "r_name")
    .agg(
        F.sum("net_revenue").alias("total_revenue"),
        F.countDistinct("o_orderkey").alias("num_orders"),
        F.avg("l_discount").alias("avg_discount"),
        F.avg("is_late").alias("late_rate"),
        F.avg("l_quantity").alias("avg_quantity"),
    )
    .withColumn("avg_order_value", F.col("total_revenue") / F.col("num_orders"))
)

def repeated_actions(df):
    df.count()
    df.agg(F.avg("total_revenue")).collect()
    df.filter(F.col("num_orders") >= 10).count()

_, t_no_cache = timed("3 actions WITHOUT cache", lambda: repeated_actions(customer_features))

In [ ]:
cache_ok = True
try:
    customer_features.cache()
    _, t_materialize = timed("Materialize cache (1st action pays the full cost)", lambda: customer_features.count())
    _, t_cached = timed("3 actions WITH cache", lambda: repeated_actions(customer_features))
    print(f"\nSpeed-up on repeated actions: {t_no_cache / t_cached:.1f}x")
except Exception as e:
    cache_ok = False
    print("df.cache() is not available on this compute:", type(e).__name__, str(e)[:200])
    print("On serverless it needs environment version 5+ and the 'DataFrame caching' preview enabled.")

## 6. Transformations (lazy) vs actions (eager)

- **Transformations** (`filter`, `withColumn`, `groupBy`, `join`) return a new DataFrame **instantly**. Spark just adds a step to the plan.
- **Actions** (`count`, `collect`, `show`, `display`, `write`) make Spark optimize the plan and **run** it.

**Why Spark is lazy:** seeing the *whole* plan before running lets Catalyst reorder filters, prune columns and pick join strategies. That's impossible if each line ran immediately.

In [ ]:
# Step 1: chain several transformations
t0 = time.perf_counter()
lazy_df = (
    raw["lineitem"]
    .filter(F.col("l_quantity") > 45)
    .withColumn("net_revenue", F.col("l_extendedprice") * (1 - F.col("l_discount")))
    .groupBy("l_shipmode")
    .agg(F.round(F.sum("net_revenue") / 1e6, 2).alias("revenue_millions"))
)
t_transform = time.perf_counter() - t0
print(f"Transformations took {t_transform * 1000:.1f} ms -> returned a {type(lazy_df).__name__}, no data read")

# Step 2: the plan exists, but no job has run
lazy_df.explain()

# Step 3: an action triggers the actual work
t0 = time.perf_counter()
rows = lazy_df.collect()
t_action = time.perf_counter() - t0
print(f"\nAction (collect) took {t_action:.2f} s -> {len(rows)} rows returned")
for r in rows:
    print(f"  {r['l_shipmode']:<8} {r['revenue_millions']:>12,.2f} M")

## 7. Write results (partitioned Parquet)

**Why `partitionBy("ship_year")`:** Spark writes one folder per year (`ship_year=1995/`, ...). A later query filtering on year reads **only that folder**. This is **partition pruning**.
**Why `repartition("ship_year")` first:** it puts each year's rows in one task, so each folder gets one file instead of many tiny ones. (We drop the `orderBy` before writing: sorting is a full shuffle that adds nothing to stored files.)

In [ ]:
(region_year_agg
    .repartition("ship_year")
    .write.mode("overwrite")
    .partitionBy("ship_year")
    .parquet(f"{OUTPUT_PATH}/region_year_summary"))

top_nations.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/top_nations_by_region")
ship_mode_perf.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/ship_mode_performance")
material_profit.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/material_profit")

display(dbutils.fs.ls(f"{OUTPUT_PATH}/region_year_summary"))

### Read back and confirm partition pruning

In the plan, look for **`PartitionFilters: [... (ship_year = 1996)]`**: Spark only opens the `ship_year=1996` folder.

In [ ]:
summary_1996 = (spark.read.parquet(f"{OUTPUT_PATH}/region_year_summary")
                .filter(F.col("ship_year") == 1996))
summary_1996.explain(mode="formatted")
display(summary_1996.orderBy(F.desc("revenue_millions")))

## 8. MLlib: KMeans customer segmentation (optional)

**Goal:** group customers by buying behaviour (revenue, order count, order size, discount use, late-delivery exposure).

| Step | Why |
|---|---|
| `VectorAssembler` | MLlib models expect all features in one vector column |
| `StandardScaler` | KMeans uses distances; without scaling, `total_revenue` (millions) would swamp `late_rate` (0-1) |
| Try k = 3, 4, 5, pick best **silhouette** | Silhouette (-1 to 1) measures how well-separated clusters are |
| Uses cached `customer_features` | Each `fit()` makes several passes over the data, a direct payoff of Section 5 |

Requires serverless **environment version 4+** (Spark ML support).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

FEATURES = ["total_revenue", "num_orders", "avg_order_value", "avg_discount", "late_rate", "avg_quantity"]

ml_input = customer_features.select("c_custkey", "c_mktsegment", "r_name", *FEATURES).na.drop()
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")

results = []
for k in (3, 4, 5):
    pipeline = Pipeline(stages=[
        VectorAssembler(inputCols=FEATURES, outputCol="raw_features"),
        StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True),
        KMeans(featuresCol="features", k=k, seed=42),
    ])
    model = pipeline.fit(ml_input)
    score = evaluator.evaluate(model.transform(ml_input))
    results.append((k, score, model))
    print(f"k={k}: silhouette = {score:.3f}")

best_k, best_score, best_model = max(results, key=lambda x: x[1])
print(f"\nBest k = {best_k} (silhouette {best_score:.3f})")

### Cluster profiles

Average feature values per cluster, so each segment can be described in business terms (e.g. "high-value frequent buyers").

In [ ]:
segments = best_model.transform(ml_input).withColumnRenamed("prediction", "segment")

segment_profile = (
    segments.groupBy("segment")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("total_revenue"), 0).alias("avg_total_revenue"),
        F.round(F.avg("num_orders"), 1).alias("avg_num_orders"),
        F.round(F.avg("avg_order_value"), 0).alias("avg_order_value"),
        F.round(F.avg("avg_discount") * 100, 2).alias("avg_discount_pct"),
        F.round(F.avg("late_rate") * 100, 1).alias("late_pct"),
    )
    .orderBy(F.desc("avg_total_revenue"))
)
display(segment_profile)

segment_profile.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/customer_segment_profile")

In [ ]:
# Free the cached data
if cache_ok:
    customer_features.unpersist()
print("Pipeline complete. Outputs written to", OUTPUT_PATH)